In [1]:
from src.train.trainer import Trainer, ConfigManager
from src.fine_tuning.tuner import Tuner
from src.prediction.predict_items import PredictUserAllItems
from src.utils.utils import read_yaml
from typing import Literal
from src.utils.utils import load_artifact
import pandas as pd
import os

In [2]:
MODEL_TYPE = 'DEEP_MATRIX_FACTORIZATION'
DATASET_TYPE = 'USER_ITEM'
DATASIZE = '100K'

In [3]:
model_config_path = 'config/models_config.yaml'
data_dir_config_path = 'config/config.yaml'
finetuning_config_path = 'config/finetuning_config.yaml'
dataset_config_path = 'config/dataset_config.yaml'
hyperparams_config_path = 'config/hyperparams_config.yaml'


model_config = ConfigManager(model_config_path, config_settings=MODEL_TYPE+'_CONFIG')
data_dir_config = ConfigManager(data_dir_config_path)
finetuning_config = ConfigManager(finetuning_config_path, MODEL_TYPE)
dataset_config = ConfigManager(dataset_config_path, config_settings=DATASET_TYPE+'_DATASET_CONFIG')
hyperparams_config = ConfigManager(hyperparams_config_path)

In [7]:
def test_model(model_type: Literal['DEEP_MATRIX_FACTORIZATION'], dataset_type: Literal['USER_ITEM'], datasize: Literal['100K']):
    # Load model configuration
    model_config = ConfigManager(model_config_path, config_settings=model_type+'_CONFIG')
    
    # Load dataset configuration
    dataset_config = ConfigManager(dataset_config_path, config_settings=dataset_type+'_DATASET_CONFIG')
    
    # Load hyperparameters configuration
    hyperparams_config = ConfigManager(hyperparams_config_path)

    # load data dir configuration
    data_dir_config = ConfigManager(data_dir_config_path)

    # Load finetuning configuration
    finetuning_config = ConfigManager(finetuning_config_path)

    # Initialize the trainer
    trainer = Trainer(model_type, 
                      dataset_type.lower(), 
                      model_config=model_config, 
                      hyperparams_config=hyperparams_config,
                      data_dir_config=data_dir_config,
                      dataset_config=dataset_config,)
    
    # Train the model
    trainer.train(max_epochs=1)
    print('Trainer Process check.')
    
    # Fine-tune the model
    tuner = Tuner('test', 1, model_type, dataset_type.lower(), finetuning_config, model_config,
                  hyperparams_config, data_dir_config, dataset_config)
    tuner.search_params(max_epochs=1)
    print('Finetuner process Check')
    
    # Predict items for a user
    movies_df_path = os.path.join('data', '100k', 'raw', 'movies.csv')
    artifact_path = data_dir_config()['ARTIFACTS']['ENCODER_PATH']
    n_movies = model_config()['n_items']
    movies_df = pd.read_csv(movies_df_path)
    movie_encoder = load_artifact(artifact_path)
    
    predictor = PredictUserAllItems(trainer.model, n_movies, movie_encoder, movies_df)
    predictions = predictor.predict(user_id=12)  # Example user_id
    preds_df = predictor.get_predicted_items(predictions)
    print('Prediction Component Check')

    return predictions, preds_df

In [8]:
MODEL_TYPE = 'DEEP_MATRIX_FACTORIZATION'
DATASET_TYPE = 'USER_ITEM'
DATASIZE = '100K'

preds, preds_df = test_model(model_type=MODEL_TYPE, dataset_type=DATASET_TYPE, datasize=DATASET_TYPE)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name    | Type                    | Params | Mode  | FLOPs
--------------------------------------------------------------------
0 | model   | DeepMatrixFactorization | 678 K  | train | 0    
1 | loss_fn | MaskedMSELoss           | 0      | train | 0    
--------------------------------------------------------------------
678 K     Trainable params
0         Non-trainable params
678 K     Total params
2.713     Total estimated model params size (MB)
15        Modules in train mode


Initializing Model
Dataset already downloaded and extracted.
data in final directory: True
Data already in local path.


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

c:\Users\ernes\miniconda3\envs\venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
c:\Users\ernes\miniconda3\envs\venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.
c:\Users\ernes\miniconda3\envs\venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:429: Consider setting `persistent_workers=True` in 'train_dataloader' to speed up the dataloader worker initialization.


Training: |          | 0/? [00:00<?, ?it/s]

...

Dataset already downloaded and extracted.
data in final directory: True
Data already in local path.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=1` reached.


[I 2026-09-03 12:03:16,687] Trial 0 finished with value: 2.051551103591919 and parameters: {'user_embedding_dim': 147, 'item_embedding_dim': 48, 'hidden_dims_nlist': 2, 'hidden_dims_1': 120, 'hidden_dims_2': 192, 'dropout_rate': 0.2740918785419214, 'lr': 0.009258854859143498, 'weight_decay': 0.008394636259234813, 'batch_size': 384}. Best is trial 0 with value: 2.051551103591919.
Finetuner process Check
Prediction Component Check


In [7]:
import torch
import pandas as pd
movie_df = pd.read_csv('data/100K/raw/movies.csv')


values, indices = torch.topk(torch.tensor(preds), 5)

indices

tensor([5199,  768, 3701, 8722, 1901])

In [8]:
movie_encoder = load_artifact('artifacts/movie_encoder.pkl')

In [9]:
indices = indices.tolist()
decoded_indices = [movie_encoder.decode_id(idx) for idx in indices]


In [10]:
d = pd.DataFrame({"movieId": decoded_indices, "scores": values.tolist()})

d.merge(movie_df, on='movieId', how='inner')

,movieId,scores,title,genres
0,8492,52.864868,"Christmas Carol, A (Scrooge) (1951)",Drama|Fantasy
1,1009,46.171139,Escape to Witch Mountain (1975),Adventure|Children|Fantasy
2,5110,44.993576,Super Troopers (2001),Comedy|Crime|Mystery
3,127164,44.309532,"What Happened, Miss Simone? (2015)",Documentary
4,2525,43.622463,Alligator (1980),Action|Horror|Sci-Fi


In [11]:
decoded_indices

[8492, 1009, 5110, 127164, 2525]